In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pinecone-client transformers sentence-transformers datasets langchain bitsandbytes accelerate torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.6/396.6 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.5/290.5 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 5.2 MB/s eta 0:00:00


In [ ]:
import json

global HF_TOKEN
global PINECONE_KEY

def load_api_keys(config_path):
    try:
        with open(config_path, 'r') as f:
            config_data = json.load(f)

        global HF_TOKEN
        global PINECONE_KEY
        HF_TOKEN = config_data.get("HF_TOKEN")
        PINECONE_KEY = config_data.get("PINECONE")

    except FileNotFoundError:
        print(f"Error: The file {config_path} was not found. Please check the file path.")
    except json.JSONDecodeError:
        print(f"Error: The file {config_path} is not a valid JSON file.")

config_path = "/content/drive/MyDrive/LLM_Assignments/config.json"
load_api_keys(config_path)

print("HF_TOKEN:", HF_TOKEN)
print("PINECONE_KEY:", PINECONE_KEY)


In [5]:
!huggingface-cli login --token $HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [6]:
import os
import random
import string
import pinecone
import warnings
from datasets import load_dataset
from langchain import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.vectorstores import Pinecone
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from transformers import BitsAndBytesConfig

warnings.filterwarnings('ignore')


**Quantization**

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

**Llama 3.1**

In [8]:
#LLAMA 3.1
llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)



tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

**OpenHathi**

In [ ]:
# OpenHathi
hathi_tokenizer = AutoTokenizer.from_pretrained("sarvamai/OpenHathi-7B-Hi-v0.1-Base")

hathi_tokenizer.pad_token = hathi_tokenizer.eos_token


hathi_model = AutoModelForCausalLM.from_pretrained(
    "sarvamai/OpenHathi-7B-Hi-v0.1-Base",
    quantization_config=bnb_config,
    device_map="auto"
)

## **PART 1**

In [9]:
import os
from pinecone import Pinecone, ServerlessSpec


# Embedding Model
embedding_model_id = 'sentence-transformers/all-MiniLM-L6-v2'

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_id,
    model_kwargs={'device':'cuda'},
    encode_kwargs={'device': 'cuda', 'batch_size': 16}
)

print(PINECONE_KEY)

pc = Pinecone(api_key=PINECONE_KEY)

index_name = 'llm-course-asg1'

'''

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

'''

index = pc.Index(index_name)
index.describe_index_stats()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

29b7a19a-3c62-4d08-a677-6621f3fb0669


{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 120000}},
 'total_vector_count': 120000}

In [ ]:
# Ag News - dataset
dataset = load_dataset('ag_news', split='train')

label_names = dataset.features['label'].names

data = dataset.to_pandas()
batch_size = 16

for i in range(0, len(data), batch_size):
    i_end = min(len(data), i + batch_size)
    batch = data.iloc[i:i_end]

    ids = [f"{i}" for i in range(i, i_end)]
    texts = [x['text'] for i, x in batch.iterrows()]

    embeddings = embedding_model.embed_documents(texts)

    meta_data = [{
        'text': x['text'],
        'label': label_names[x['label']]
    } for i, x in batch.iterrows()]

    # Upsert vectors into Pinecone index
    index.upsert(vectors=zip(ids, embeddings, meta_data))


**Self-Consistency Prompts:**

In [ ]:
def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "Who was the leader who became US's first president?"

llama_response = generate_response(llama_model, llama_tokenizer, prompt)
print("\n LLAMA 3.1 Response:", llama_response)


hathi_response = generate_response(hathi_model, hathi_tokenizer, prompt)
print("\n OpenHathi Response:", hathi_response)



 LLAMA 3.1 Response: Who was the leader who became US's first president? | The Indian Express
Who was the leader who became US's first president? | The Indian Express
Who was the leader who became US's first president?
George Washington was the leader who became the first President of the United States. He was inaugur

 OpenHathi Response: Who was the leader who became US's first president?
 संतुलित उत्तरः
---
The leader who became the first president of the United States was George Washington. वह एक बहुत ही महत्वपूर्ण व्यक्ति थे जिन्होंने देश के लिए बहुत काम किया और उन्हें बहुत पसंद किया जाता था।


In [ ]:
def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "Which prominent figure served as the first president of the US?"
llama_response = generate_response(llama_model, llama_tokenizer, prompt)
print("\n LLAMA 3.1 Response:", llama_response)


hathi_response = generate_response(hathi_model, hathi_tokenizer, prompt)
print("\n OpenHathi Response:", hathi_response)



 LLAMA 3.1 Response: Which prominent figure served as the first president of the US? A) George Washington B) Thomas Jefferson C) Abraham Lincoln D) Franklin D. Roosevelt
The correct answer is A) George Washington. George Washington was the first president of the United States, serving from 1789 to 1797. He

 OpenHathi Response: Which prominent figure served as the first president of the US?
 संतुलित उत्तरः
---
The first president of the United States was George Washington. वह एक प्रमुख व्यक्ति थे जिन्होंने देश के पहले राष्ट्रपति के रूप में कार्य किया।


In [ ]:
def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "Who was the first person to hold the office of President in US?"
llama_response = generate_response(llama_model, llama_tokenizer, prompt)
print("\n LLAMA 3.1 Response:", llama_response)


hathi_response = generate_response(hathi_model, hathi_tokenizer, prompt)
print("\n OpenHathi Response:", hathi_response)



 LLAMA 3.1 Response: Who was the first person to hold the office of President in US? Who was the first person to hold the office of President in US?
The first person to hold the office of President in the United States was George Washington. He was inaugurated as the first President of the United States on April 30, 178

 OpenHathi Response: Who was the first person to hold the office of President in US?
 संतुलित बजट का क्या अर्थ है?
What is the difference between a presidential election and a congressional election?
संयुक्त राज्य अमेरिका के राष्ट्रपति का कार्यकाल कितना लंबा होता है?
What is the difference between a presidential election and


**Fact-Checking Prompts:**

In [ ]:
def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "When did World War II end?"
llama_response = generate_response(llama_model, llama_tokenizer, prompt)
print("\n LLAMA 3.1 Response:", llama_response)


hathi_response = generate_response(hathi_model, hathi_tokenizer, prompt)
print("\n OpenHathi Response:", hathi_response)



 LLAMA 3.1 Response: When did World War II end? World War II officially ended on September 2, 1945, when Japan formally surrendered to the Allied Powers on board the USS Missouri in Tokyo Bay, Japan. The war in Europe had ended earlier, on May 8, 1945,

 OpenHathi Response: When did World War II end?
 संतुलित उत्तरः
---
World War II ended on September 2, 1945, with the signing of the Japanese Instrument of Surrender by Japanese Emperor Hirohito. यह आत्मसमर्पण 2 सितंबर,


In [ ]:

def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "Who discovered penicillin?"
llama_response = generate_response(llama_model, llama_tokenizer, prompt)
print("\n LLAMA 3.1 Response:", llama_response)


hathi_response = generate_response(hathi_model, hathi_tokenizer, prompt)
print("\n OpenHathi Response:", hathi_response)



 LLAMA 3.1 Response: Who discovered penicillin? | Who discovered penicillin?
Alexander Fleming discovered penicillin in 1928. He was a Scottish scientist, biologist, and pharmacologist who was working in his laboratory at St. Mary's Hospital in London, England. Fleming had been studying

 OpenHathi Response: Who discovered penicillin?
 संतुलित उत्तरः
---
पेनिसिलिन की खोज 1928 में स्कॉटिश वैज्ञानिक अलेक्जेंडर फ्लेमिंग ने की थी। He was working at the University of Edinburgh at


In [ ]:

def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "What is the distance between the Earth and the Sun?"
llama_response = generate_response(llama_model, llama_tokenizer, prompt)
print("\n LLAMA 3.1 Response:", llama_response)


hathi_response = generate_response(hathi_model, hathi_tokenizer, prompt)
print("\n OpenHathi Response:", hathi_response)



 LLAMA 3.1 Response: What is the distance between the Earth and the Sun? The average distance between the Earth and the Sun is approximately 149.6 million kilometers (92.96 million miles). This distance is called an astronomical unit, or AU. The distance between the Earth and the Sun varies slightly throughout the year due to

 OpenHathi Response: What is the distance between the Earth and the Sun?
 संतुलित समीकरणः
formula_1
सूत्र _ 2
formula_3
सूत्र _ 4
formula_5
सूत्र _ 6
formula_7
सूत्र _ 8
formula_


**RAG PIPELINE**

OpenHathi + RAG

In [ ]:
from langchain.vectorstores import Pinecone
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Text-Generation Pipeline
pipe = pipeline(
    'text-generation',
    model=hathi_model,
    tokenizer=hathi_tokenizer,
    device_map='auto',
    max_new_tokens=50,
    temperature=0.2,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=hathi_tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=pipe)

# Pinecone Vectorstore for Retrieval
embedding_function = embedding_model.embed_query

vectorstore = Pinecone(
    index=index,
    embedding=embedding_function,
    text_key='text'
)



In [ ]:
query = "Who was the first person to hold the office of President in US?"

response = vectorstore.similarity_search(query, k=5)

print(f"Number of Responses Returned: {len(response)}")

# RAG Pipeline
rag_pipeline = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=vectorstore.as_retriever()
)

response_with_rag = rag_pipeline(query)

answer = response_with_rag['result']
print("Answer:", answer)


Number of Responses Returned: 5
Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

First Lady to Hail Bush on His Leadership (AP) AP - Republican National Convention co-stars Laura Bush and Arnold Schwarzenegger commended President Bush to the country Tuesday for four more years in office, praising him for unflinching leadership in a time of national testing. "I am so proud of the way George has led our country with strength and conviction" in the war on terror, the first lady planned to say.

The Matrix Online finds its voice  quot;80 of Americans could correctly name the first President Bush #39;s pet dog Millie, but only 15 knew that both President Bush and the challenger, Bill Clinton, favoured the death penalty quot; Link.

Scientists to Flesh Out George Washington's Appearance (Reuters) Reuters - Americans know George\Washington as the dour founding father with

In [ ]:
query = "What is the distance between the Earth and the Sun?"

response = vectorstore.similarity_search(query, k=5)

print(f"Number of Responses Returned: {len(response)}")

# RAG Pipeline
rag_pipeline = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=vectorstore.as_retriever()
)

response_with_rag = rag_pipeline(query)

answer = response_with_rag['result']
print("Answer:", answer)


Number of Responses Returned: 5
Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Sun Is Closer Than We Thought The solar particles are sealed inside the Genesis sample return capsule, a disk about the size of a truck tire, that will drift toward Earth #39;s surface beneath a parachute.

Calculating the Precise Distance to Doomsday Today's close passage of the relatively large asteroid, Toutatis, has presented astronomers with a new ruler to measure solar system distances. By using two telescopes separated by hundreds of miles, the classic parallax method validates the asteroid's orbital path...

Asteroid (4179) Toutatis to Pass Closely By Earth on Wednesday &lt;b&gt;...&lt;/b&gt; Toutatis, a potato-shaped asteroid about 4.6 km (3 miles) in its longest extent, will pass within 1,550,000 km (963,000 miles) of the Earth #39;s center on Wednesday, September 29, 2004 - 

Llama + Rag

In [10]:
from langchain.vectorstores import Pinecone
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Text-Generation Pipeline
pipe = pipeline(
    'text-generation',
    model=llama_model,
    tokenizer=llama_tokenizer,
    device_map='auto',
    max_new_tokens=50,
    temperature=0.2,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=llama_tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=pipe)

# Pinecone Vectorstore for Retrieval
embedding_function = embedding_model.embed_query

vectorstore = Pinecone(
    index=index,
    embedding=embedding_function,
    text_key='text'
)



In [12]:
query = "Who was the leader who became US's 1st president?"

response = vectorstore.similarity_search(query, k=5)

print(f"Number of Responses Returned: {len(response)}")

# RAG Pipeline
rag_pipeline = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=vectorstore.as_retriever()
)

response_with_rag = rag_pipeline(query)

answer = response_with_rag['result']
print("Answer:", answer)


Number of Responses Returned: 5
Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

First Lady to Hail Bush on His Leadership (AP) AP - Republican National Convention co-stars Laura Bush and Arnold Schwarzenegger commended President Bush to the country Tuesday for four more years in office, praising him for unflinching leadership in a time of national testing. "I am so proud of the way George has led our country with strength and conviction" in the war on terror, the first lady planned to say.

The Matrix Online finds its voice  quot;80 of Americans could correctly name the first President Bush #39;s pet dog Millie, but only 15 knew that both President Bush and the challenger, Bill Clinton, favoured the death penalty quot; Link.

Topics A, B and C in Run-up to Debate: Jobs, Jobs and Jobs Senator John Kerry will assail George Bush tonight as the first president since H

## **PART 2**

In [ ]:
from datasets import load_dataset
import pandas as pd

dbpedia = load_dataset('dbpedia_14', split='train')


Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

**Classification Model**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import numpy as np
import random
import torch

label_encoder = LabelEncoder()
dbpedia_labels = label_encoder.fit_transform(dbpedia['label'])

def create_prompt(entity):
    return f"Question: Provide detailed information about '{entity}'?\nAnswer:"

def extract_final_token_embedding(prompt, model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to('cuda')

    with torch.no_grad():
        outputs = model(input_ids, output_hidden_states=True)

    last_hidden_state = outputs.hidden_states[-1]
    final_token_embedding = last_hidden_state[:, -1, :]

    return final_token_embedding.cpu().numpy()

num_samples = 500

total_samples = len(dbpedia['title'])
random_indices = random.sample(range(total_samples), num_samples)

embeddings = []
sample_entities = [dbpedia['title'][i] for i in random_indices]
sample_labels = [dbpedia_labels[i] for i in random_indices]

for entity in sample_entities:
    prompt = create_prompt(entity)
    embedding = extract_final_token_embedding(prompt, llama_model, llama_tokenizer)
    embeddings.append(embedding)

embeddings = np.vstack(embeddings)  # Stack embeddings into a matrix

X_train, X_test, y_train, y_test = train_test_split(embeddings, sample_labels, test_size=0.2, random_state=42)

classifier = RandomForestClassifier(n_estimators=100, random_state=42)

classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print("Sample Predictions vs Actual Labels:")
for pred, actual in zip(y_pred[:10], y_test[:10]):
    actual_label = label_encoder.inverse_transform([actual])[0]
    predicted_label = label_encoder.inverse_transform([pred])[0]
    print(f"Predicted: {predicted_label}, Actual: {actual_label}")


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Accuracy: 0.67
Sample Predictions vs Actual Labels:
Predicted: 1, Actual: 6
Predicted: 12, Actual: 11
Predicted: 9, Actual: 9
Predicted: 9, Actual: 12
Predicted: 6, Actual: 6
Predicted: 12, Actual: 11
Predicted: 13, Actual: 11
Predicted: 10, Actual: 10
Predicted: 2, Actual: 4
Predicted: 8, Actual: 8


In [ ]:
import numpy as np
import random
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

class LLMClassifier:
    def __init__(self, llama_model, llama_tokenizer):
        self.llama_model = llama_model
        self.llama_tokenizer = llama_tokenizer
        self.label_encoder = LabelEncoder()

    def encode_labels(self, labels):
        return self.label_encoder.fit_transform(labels)

    def create_prompt(self, entity):
        return f"Question: Provide detailed information about '{entity}'?\nAnswer:"

    def extract_layer_embedding(self, prompt, layer_type="final"):
        inputs = self.llama_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to('cuda')

        with torch.no_grad():
            outputs = self.llama_model(input_ids, output_hidden_states=True)

        layer_index = {
            "first": 1,
            "middle": len(outputs.hidden_states) // 2,
            "final": -1
        }.get(layer_type, -1)

        layer_embedding = outputs.hidden_states[layer_index][:, -1, :]
        return layer_embedding.cpu().numpy()

    def extract_embeddings(self, titles, labels, layer, num_samples=500):
        random_indices = random.sample(range(len(titles)), num_samples)
        sample_entities = [titles[i] for i in random_indices]
        sample_labels = [labels[i] for i in random_indices]

        embeddings = [
            self.extract_layer_embedding(self.create_prompt(entity), layer_type=layer)
            for entity in sample_entities
        ]

        return np.vstack(embeddings), sample_labels

    def train_and_evaluate_classifier(self, embeddings, labels):
        X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)
        classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        classifier.fit(X_train, y_train)
        y_pred = classifier.predict(X_test)
        return accuracy_score(y_test, y_pred)

    def compare_layers_performance(self, dbpedia, num_samples=500):
        encoded_labels = self.encode_labels(dbpedia['label'])
        layers = ['first', 'middle', 'final']
        accuracy_results = {}

        for layer in layers:
            print(f"Processing {layer} layer embeddings...")
            embeddings, sample_labels = self.extract_embeddings(dbpedia['title'], encoded_labels, layer, num_samples)
            accuracy = self.train_and_evaluate_classifier(embeddings, sample_labels)
            accuracy_results[layer] = accuracy
            print(f"Accuracy for {layer} layer: {accuracy}")

            # Save embeddings
            np.save(f"/content/drive/MyDrive/LLM_Assignments/{layer}_layer_embeddings_classification.npy", embeddings)

        return accuracy_results

def main(dbpedia, llama_model, llama_tokenizer):
    classifier = LLMClassifier(llama_model, llama_tokenizer)
    accuracy_results = classifier.compare_layers_performance(dbpedia, num_samples=500)

    print("\nAccuracy comparison across layers:")
    for layer, acc in accuracy_results.items():
        print(f"{layer.capitalize()} Layer Accuracy: {acc}")

if __name__ == "__main__":

    main(dbpedia, llama_model, llama_tokenizer)

Processing first layer embeddings...
Accuracy for first layer: 0.4
Processing middle layer embeddings...
Accuracy for middle layer: 0.62
Processing final layer embeddings...
Accuracy for final layer: 0.6

Accuracy comparison across layers:
First Layer Accuracy: 0.4
Middle Layer Accuracy: 0.62
Final Layer Accuracy: 0.6


**Linear Regression**

In [ ]:
import numpy as np
import random
import torch
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

class LLMRegressor:
    def __init__(self, llama_model, llama_tokenizer):
        self.llama_model = llama_model
        self.llama_tokenizer = llama_tokenizer

    def create_prompt(self, entity):
        return f"Question: Provide detailed information about the movie '{entity}' rating? \nAnswer:"

    def extract_layer_embedding(self, prompt, layer_type="final"):
        inputs = self.llama_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to('cuda')

        with torch.no_grad():
            outputs = self.llama_model(input_ids, output_hidden_states=True)

        layer_index = {
            "first": 1,
            "middle": len(outputs.hidden_states) // 2,
            "final": -1
        }.get(layer_type, -1)

        layer_embedding = outputs.hidden_states[layer_index][:, -1, :]
        return layer_embedding.cpu().numpy()

    def extract_embeddings(self, movies_df, layer, num_samples=500):
        total_samples = len(movies_df)
        random_indices = random.sample(range(total_samples), num_samples)

        sample_entities = movies_df['Series_Title'].iloc[random_indices].tolist()
        sample_labels = movies_df['IMDB_Rating'].iloc[random_indices].tolist()

        embeddings = [
            self.extract_layer_embedding(self.create_prompt(entity), layer_type=layer)
            for entity in sample_entities
        ]

        return np.vstack(embeddings), sample_labels

    def train_and_evaluate_regressor(self, embeddings, labels):
        X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        y_pred = regressor.predict(X_test)
        return mean_squared_error(y_test, y_pred)

    def compare_layers_performance(self, movies_df, num_samples=500):
        layers = ['first', 'middle', 'final']
        mse_results = {}

        for layer in layers:
            print(f"Processing {layer} layer embeddings...")
            embeddings, sample_labels = self.extract_embeddings(movies_df, layer, num_samples)
            mse = self.train_and_evaluate_regressor(embeddings, sample_labels)
            mse_results[layer] = mse
            print(f"Mean Squared Error (MSE) for {layer} layer: {mse}")

            # Save embeddings
            np.save(f"/content/drive/MyDrive/LLM_Assignments/{layer}_layer_embeddings_regression.npy", embeddings)

        return mse_results

def load_and_preprocess_data(dataset_path):
    movies_df = pd.read_csv(dataset_path)
    return movies_df.dropna(subset=['Released_Year', 'Runtime', 'IMDB_Rating'])

def main(dataset_path, llama_model, llama_tokenizer):
    movies_df = load_and_preprocess_data(dataset_path)
    regressor = LLMRegressor(llama_model, llama_tokenizer)
    mse_results = regressor.compare_layers_performance(movies_df, num_samples=500)

    print("\nMean Squared Error comparison across layers:")
    for layer, mse in mse_results.items():
        print(f"{layer.capitalize()} Layer MSE: {mse}")

if __name__ == "__main__":
    # IMBD dataset
    dataset_path = '/content/drive/MyDrive/LLM_Assignments/imdb_top_1000.csv'

    main(dataset_path, llama_model, llama_tokenizer)

Processing first layer embeddings...
Mean Squared Error (MSE) for first layer: 0.2590393859863281
Processing middle layer embeddings...
Mean Squared Error (MSE) for middle layer: 0.08636047973632811
Processing final layer embeddings...
Mean Squared Error (MSE) for final layer: 0.08555673828125002

Mean Squared Error comparison across layers:
First Layer MSE: 0.2590393859863281
Middle Layer MSE: 0.08636047973632811
Final Layer MSE: 0.08555673828125002
